# 📦 Notebook 1 — Setup & Installation
## ContractIQ — Environment Setup
Run each cell top to bottom. Green tick = success.


## Step 1 — Install All Required Packages

In [ ]:
# Run this cell first — installs everything
# This may take 3-5 minutes on first run

import subprocess, sys

packages = [
    "groq",
    "pymupdf",
    "pdfplumber",
    "chromadb",
    "sentence-transformers",
    "langchain",
    "langchain-community",
    "langchain-groq",
    "sqlalchemy",
    "python-dotenv",
    "streamlit",
    "plotly",
    "pandas",
    "openpyxl",
    "reportlab",
    "apscheduler",
    "spacy",
    "python-multipart",
    "fastapi",
    "uvicorn",
    "nbformat",
    "ipywidgets",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {pkg}")
    else:
        print(f"  ⚠️  {pkg} — {result.stderr[:80]}")

print("\n✅ All packages processed!")

## Step 2 — Download spaCy Language Model

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else "")
print("✅ spaCy model ready!" if result.returncode == 0 else f"⚠️ {result.stderr[:200]}")

## Step 3 — Create Your .env File (API Key Goes Here)

In [ ]:
import os

env_path = ".env"

# Check if .env already exists
if os.path.exists(env_path):
    print("✅ .env file already exists!")
    print("   Make sure it contains: GROQ_API_KEY=your_key_here")
else:
    # Create the .env file template
    with open(env_path, "w") as f:
        f.write("# ContractIQ Environment Variables\n")
        f.write("# Paste your Groq API key below (no quotes needed)\n")
        f.write("GROQ_API_KEY=paste_your_groq_key_here\n")
    
    print("✅ .env file created!")
    print("")
    print("👉 ACTION REQUIRED:")
    print("   1. Open the .env file in your project folder")
    print("   2. Replace 'paste_your_groq_key_here' with your actual Groq API key")
    print("   3. Save the file")
    print("   4. Come back and run Step 4")

## Step 4 — Verify API Key Works

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads from .env file

api_key = os.getenv("GROQ_API_KEY")

if not api_key or api_key == "paste_your_groq_key_here":
    print("❌ API key not set!")
    print("   Open .env file and paste your Groq API key")
else:
    print(f"✅ API key loaded: {api_key[:8]}...{api_key[-4:]}")
    
    # Test the connection
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model="llama3-70b-8192",
            messages=[{"role": "user", "content": "Say: ContractIQ is ready!"}],
            max_tokens=20
        )
        print(f"✅ Groq API working: {response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Groq API error: {e}")

## Step 5 — Create Project Folder Structure

In [ ]:
import os

folders = [
    "contracts",        # uploaded contract PDFs go here
    "database",         # SQLite database files
    "vectorstore",      # ChromaDB vector storage
    "exports",          # generated reports
    "test_contracts",   # test PDFs from generator
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"  ✅ /{folder}")

print("\n✅ Project structure ready!")

## Step 6 — Create the Database

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime, Float
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime

# Create database
engine = create_engine("sqlite:///database/contractiq.db", echo=False)
Base = declarative_base()

# ── Table 1: Documents ──────────────────────────────────────
class Document(Base):
    __tablename__ = "documents"
    id           = Column(Integer, primary_key=True)
    filename     = Column(String(255))
    filepath     = Column(String(500))
    upload_date  = Column(DateTime, default=datetime.now)
    total_pages  = Column(Integer, default=0)
    total_chunks = Column(Integer, default=0)
    status       = Column(String(50), default="uploaded")  # uploaded/processing/done/error

# ── Table 2: Obligations ────────────────────────────────────
class Obligation(Base):
    __tablename__ = "obligations"
    id           = Column(Integer, primary_key=True)
    doc_id       = Column(Integer)
    doc_name     = Column(String(255))
    ob_type      = Column(String(100))  # SLA, Payment, Deadline, etc.
    text         = Column(Text)
    clause_ref   = Column(String(100))  # e.g. "Section 3.2"
    deadline     = Column(String(200))  # e.g. "45 days", "2024-12-31"
    risk_level   = Column(String(20))   # HIGH / MEDIUM / LOW
    created_at   = Column(DateTime, default=datetime.now)

# ── Table 3: Conflicts ──────────────────────────────────────
class Conflict(Base):
    __tablename__ = "conflicts"
    id           = Column(Integer, primary_key=True)
    ob1_id       = Column(Integer)
    ob2_id       = Column(Integer)
    doc1_name    = Column(String(255))
    doc2_name    = Column(String(255))
    conflict_type= Column(String(100))
    ob1_text     = Column(Text)
    ob2_text     = Column(Text)
    explanation  = Column(Text)
    severity     = Column(String(20))   # HIGH / MEDIUM / LOW
    created_at   = Column(DateTime, default=datetime.now)

# ── Table 4: Deadlines ──────────────────────────────────────
class Deadline(Base):
    __tablename__ = "deadlines"
    id           = Column(Integer, primary_key=True)
    doc_name     = Column(String(255))
    obligation   = Column(Text)
    deadline_str = Column(String(200))
    days_remaining = Column(Integer, default=999)
    risk_level   = Column(String(20))
    created_at   = Column(DateTime, default=datetime.now)

# ── Table 5: Chunks (for RAG) ───────────────────────────────
class Chunk(Base):
    __tablename__ = "chunks"
    id           = Column(Integer, primary_key=True)
    doc_id       = Column(Integer)
    doc_name     = Column(String(255))
    chunk_text   = Column(Text)
    page_number  = Column(Integer, default=0)
    chunk_index  = Column(Integer, default=0)

# Create all tables
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

print("✅ Database created at: database/contractiq.db")
print("✅ 5 tables created:")
print("   → documents, obligations, conflicts, deadlines, chunks")

## ✅ Setup Complete! 

Now run the other notebooks in order:
- `02_PDF_Parser.ipynb`
- `03_Obligation_Extractor.ipynb`
- `04_Conflict_Detector.ipynb`
- `05_RAG_Engine.ipynb`

Then launch the demo: `streamlit run demo_app.py`